### Generate Cluster Bounding-Box for easier visualization

In [1]:
import pandas as pd
from scipy.spatial import ConvexHull

In [2]:
df = pd.read_csv('../data/predictions.csv')
# Convert id column to int
df['id'] = df['id'].astype(int)
df.head()

,id,hdbscan,dbscan,kmean,isolation_forest
0,4395181099,481.0,1171.0,235.0,NaN
1,4394748717,-1.0,-1.0,182.0,NaN
2,4394694699,577.0,1172.0,441.0,NaN
3,4394803790,-1.0,1084.0,436.0,NaN
4,4394803554,-1.0,1084.0,436.0,NaN


In [3]:
# Filter only id and algorithm from the columns
hdbscan_df = df[['id', 'hdbscan']]
dbscan_df = df[['id', 'dbscan']]
kmeans_df = df[['id', 'kmean']]

In [4]:
kmeans_ggp = kmeans_df.groupby('kmean').agg(
    n_points=('id', list)
)
dbscan_ggp = dbscan_df.groupby('dbscan').agg(
    n_points=('id', list)
)
hdbscan_ggp = hdbscan_df.groupby('hdbscan').agg(
    n_points=('id', list)
)

In [5]:
# Load original data to get lat and long
geo_df = pd.read_csv('../data/data_cleaned_titles.csv')
geo_df['id'] = geo_df['id'].astype(int)
geo_df = geo_df[['id', 'lat', 'long']]
geo_df.head()

# Replace ids with lat and long
def replace_ids_with_coords(id_list):
    coords = geo_df[geo_df['id'].isin(id_list)][['lat', 'long']].to_numpy()
    return coords
kmeans_ggp['n_points'] = kmeans_ggp['n_points'].apply(replace_ids_with_coords)
dbscan_ggp['n_points'] = dbscan_ggp['n_points'].apply(replace_ids_with_coords)
hdbscan_ggp['n_points'] = hdbscan_ggp['n_points'].apply(replace_ids_with_coords)

C:\Users\jhony\AppData\Local\Temp\ipykernel_25692\3101033559.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  geo_df = pd.read_csv('../data/data_cleaned_titles.csv')


In [6]:
# Drop unclustered points for DBSCAN and HDBSCAN
dbscan_ggp = dbscan_ggp.drop(index=-1, errors='ignore')
hdbscan_ggp = hdbscan_ggp.drop(index=-1, errors='ignore')

In [7]:
dbscan_ggp

,n_points
dbscan,
2.0,"[[45.84137, 4.798471], [45.84149, 4.797935], [..."
3.0,"[[45.699631, 4.941516], [45.694265, 4.940006],..."
4.0,"[[45.831266, 4.879458], [45.831266, 4.879458],..."
5.0,"[[45.730703, 4.739656], [45.730673, 4.738948],..."
6.0,"[[45.779488, 4.923328], [45.779488, 4.923328],..."
...,...
1867.0,"[[45.739167, 4.814521], [45.739167, 4.814521],..."
1868.0,"[[45.73459, 4.815572], [45.73459, 4.815572], [..."
1869.0,"[[45.741758, 4.820797], [45.741758, 4.820797],..."


In [8]:
# For each cluster of an algorithm, compute the bounding polygon of the points in that cluster.
from scipy.spatial import QhullError
def compute_bounding_polygon(points):
    if len(points) < 3:
        return None
    try:
        hull = ConvexHull(points)
        polygon = points[hull.vertices]
        return polygon.tolist()
    except QhullError:
        return None

In [9]:
# For each algorithm, determine its clusters' convex hull
def get_convex_hull_algorithm(algo_clusters):
    convex_hulls = []
    for cluster_id, row in algo_clusters.iterrows():
        points = row['n_points']
        convex_hull = compute_bounding_polygon(points)
        convex_hulls.append({
            'cluster_id': cluster_id,
            'convex_hull': convex_hull
        })
    return convex_hulls

kmeans_hulls = pd.DataFrame(get_convex_hull_algorithm(kmeans_ggp))
dbscan_hulls = pd.DataFrame(get_convex_hull_algorithm(dbscan_ggp))
hdbscan_hulls = pd.DataFrame(get_convex_hull_algorithm(hdbscan_ggp))

In [10]:
print("KMeans Convex Hulls:")
print(kmeans_hulls)
print("DBSCAN Convex Hulls:")
print(dbscan_hulls)
print("HDBSCAN Convex Hulls:")
print(hdbscan_hulls)

KMeans Convex Hulls:
     cluster_id                                        convex_hull
0           0.0  [[45.769999, 4.857716], [45.770471, 4.859068],...
1           1.0  [[45.771077, 4.83466], [45.770247, 4.834628], ...
2           2.0  [[45.836836, 4.826377], [45.836666, 4.822777],...
3           3.0  [[45.730558, 4.952496], [45.730133, 4.950113],...
4           4.0  [[45.739535, 4.81967], [45.739666, 4.818166], ...
..          ...                                                ...
495       495.0  [[45.770724, 4.987092], [45.765713, 4.985361],...
496       496.0  [[45.751601, 4.839241], [45.751261, 4.840263],...
497       497.0  [[45.773345, 4.80787], [45.774628, 4.808739], ...
498       498.0  [[45.772252, 4.829564], [45.771911, 4.828855],...
499       499.0  [[45.811003, 4.897327], [45.810775, 4.893986],...

[500 rows x 2 columns]
DBSCAN Convex Hulls:
     cluster_id                                        convex_hull
0           2.0  [[45.83747, 4.805295], [45.833102, 4.79988], [

In [11]:
# Save the convex hulls to csv files
kmeans_hulls.to_csv('../data/kmean_convex_hulls.csv', index=False)
dbscan_hulls.to_csv('../data/dbscan_convex_hulls.csv', index=False)
hdbscan_hulls.to_csv('../data/hdbscan_convex_hulls.csv', index=False)